# OGC API Features — Collections / Items CRUD + CQL2 Filter

Demonstrates OGC API — Features (Parts 1–4) against DynaStore.

## What this notebook covers

1. Catalog setup via the `public_open_data` preset.
2. Collection CRUD (create / read / replace / delete).
3. Item CRUD (create, read, replace, delete).
4. CQL2-text filter (`filter=` + `filter-lang=cql2-text`).
5. CQL2-JSON filter (`filter-lang=cql2-json`).
6. Teardown.

**Env vars:** `DYNASTORE_BASE_URL` (default `http://localhost:8080`),
`DYNASTORE_SYSADMIN_TOKEN` or `DYNASTORE_TOKEN`.

**Routes verified against** `features_service.py` `_register_routes()`:
- `POST /features/catalogs/{cat}/collections` → 201
- `GET /features/catalogs/{cat}/collections/{col}` → 200
- `PUT /features/catalogs/{cat}/collections/{col}` → 200
- `DELETE /features/catalogs/{cat}/collections/{col}` → 204
- `POST /features/catalogs/{cat}/collections/{col}/items` → 201
- `GET /features/catalogs/{cat}/collections/{col}/items` → 200
- `GET /features/catalogs/{cat}/collections/{col}/items/{id}` → 200
- `PUT /features/catalogs/{cat}/collections/{col}/items/{id}` → 200
- `DELETE /features/catalogs/{cat}/collections/{col}/items/{id}` → 204

In [ ]:
import os
import json
import httpx
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True), override=False)

BASE = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080").rstrip("/")
TOKEN = (
    os.environ.get("DYNASTORE_SYSADMIN_TOKEN")
    or os.environ.get("DYNASTORE_TOKEN")
    or ""
)

# Auto-provision token via IDP client_credentials if not set
if not TOKEN:
    _idp = (os.environ.get("IDP_PUBLIC_URL") or os.environ.get("IDP_ISSUER_URL", "")).rstrip("/")
    _cid = os.environ.get("IDP_CLIENT_ID", "")
    _sec = os.environ.get("IDP_CLIENT_SECRET", "")
    if _idp and _cid and _sec:
        try:
            _r = httpx.post(
                f"{_idp}/protocol/openid-connect/token",
                data={"grant_type": "client_credentials", "client_id": _cid, "client_secret": _sec},
                timeout=10,
            )
            if _r.status_code == 200:
                TOKEN = _r.json().get("access_token", "")
        except Exception:
            pass

HEADERS = {"Authorization": f"Bearer {TOKEN}"} if TOKEN else {}
client = httpx.Client(base_url=BASE, headers=HEADERS, timeout=60.0)

CATALOG_ID = "nb02-features-demo"
COLL_ID = "cities"

def check(r, label="", expected=(200, 201, 204)):
    msg = f"{label}: {r.status_code}"
    if r.status_code not in expected:
        msg += f" — {r.text[:300]}"
    print(msg)
    assert r.status_code in expected, msg
    return r

print(f"Base URL : {BASE}")
print(f"Catalog  : {CATALOG_ID}")
print(f"Collection: {COLL_ID}")

## 1. Catalog setup via `public_open_data` preset

Create the catalog via STAC (catalog-creation API), then apply the
`public_open_data` composite preset to wire roles, IAM, routing, STAC,
web, and admin interfaces in one call.

In [ ]:
r = client.post("/stac/catalogs", json={
    "id": CATALOG_ID,
    "title": "OGC Features Demo",
    "description": "Ephemeral catalog for the OGC API Features notebook.",
})
check(r, "Create catalog", expected=(200, 201, 409))

# Apply public_open_data composite preset at catalog scope
r = client.post(f"/configs/catalogs/{CATALOG_ID}/presets/public_open_data")
check(r, "Apply preset public_open_data", expected=(200, 201, 409))
print("Catalog ready.")

## 2. Collection CRUD

In [ ]:
# Create collection
r = client.post(f"/features/catalogs/{CATALOG_ID}/collections", json={
    "id": COLL_ID,
    "title": "World Cities",
    "description": "Sample point collection — world cities with population.",
    "license": "CC-BY-4.0",
})
check(r, "Create collection", expected=(201,))
coll = r.json()
assert coll["id"] == COLL_ID
print(f"Created: {coll['id']}")

In [ ]:
# Read collection
r = client.get(f"/features/catalogs/{CATALOG_ID}/collections/{COLL_ID}")
check(r, "Read collection")
coll = r.json()
assert coll["id"] == COLL_ID
print(f"Title: {coll.get('title')}")

In [ ]:
# Replace (PUT) collection — update description
r = client.put(f"/features/catalogs/{CATALOG_ID}/collections/{COLL_ID}", json={
    "id": COLL_ID,
    "title": "World Cities (updated)",
    "description": "Updated description.",
    "license": "CC-BY-4.0",
})
check(r, "Replace collection")
print(f"Updated title: {r.json().get('title')}")

## 3. Item CRUD

In [ ]:
# Create single item
ITEM_ID = "city-rome"
r = client.post(f"/features/catalogs/{CATALOG_ID}/collections/{COLL_ID}/items", json={
    "type": "Feature",
    "id": ITEM_ID,
    "geometry": {"type": "Point", "coordinates": [12.4964, 41.9028]},
    "properties": {"name": "Rome", "country": "IT", "population": 2873000},
})
check(r, "Create item", expected=(201,))
print(f"Created item: {r.json().get('id')}")

In [ ]:
# Bulk create via FeatureCollection
r = client.post(f"/features/catalogs/{CATALOG_ID}/collections/{COLL_ID}/items", json={
    "type": "FeatureCollection",
    "features": [
        {"type": "Feature", "id": "city-paris",
         "geometry": {"type": "Point", "coordinates": [2.3522, 48.8566]},
         "properties": {"name": "Paris", "country": "FR", "population": 2148000}},
        {"type": "Feature", "id": "city-berlin",
         "geometry": {"type": "Point", "coordinates": [13.4050, 52.5200]},
         "properties": {"name": "Berlin", "country": "DE", "population": 3645000}},
        {"type": "Feature", "id": "city-madrid",
         "geometry": {"type": "Point", "coordinates": [-3.7038, 40.4168]},
         "properties": {"name": "Madrid", "country": "ES", "population": 3223000}},
    ],
})
check(r, "Bulk create items", expected=(200, 201))
print(f"Bulk response type: {r.json().get('type')}")

In [ ]:
# List items
r = client.get(f"/features/catalogs/{CATALOG_ID}/collections/{COLL_ID}/items")
check(r, "List items")
fc = r.json()
assert fc["type"] == "FeatureCollection"
print(f"Total items returned: {len(fc.get('features', []))}")

In [ ]:
# Read single item
r = client.get(f"/features/catalogs/{CATALOG_ID}/collections/{COLL_ID}/items/{ITEM_ID}")
check(r, "Read item")
feat = r.json()
assert feat["type"] == "Feature"
assert feat["id"] == ITEM_ID
print(f"Item: id={feat['id']}  name={feat['properties'].get('name')}")

In [ ]:
# Replace (PUT) item — update population
r = client.put(f"/features/catalogs/{CATALOG_ID}/collections/{COLL_ID}/items/{ITEM_ID}", json={
    "type": "Feature",
    "id": ITEM_ID,
    "geometry": {"type": "Point", "coordinates": [12.4964, 41.9028]},
    "properties": {"name": "Rome", "country": "IT", "population": 2900000},
})
check(r, "Replace item")
print(f"Updated population: {r.json()['properties'].get('population')}")

## 4. CQL2-text filter

The features router accepts `filter=<expr>` + `filter-lang=cql2-text`
on `GET /features/catalogs/{cat}/collections/{col}/items`.

Verified against `features_service.py`: route uses `parse_ogc_query_request`
which calls `validate_filter_lang` (query.py) and routes to pygeofilter.

In [ ]:
# Filter: country = 'DE' using CQL2-text
r = client.get(
    f"/features/catalogs/{CATALOG_ID}/collections/{COLL_ID}/items",
    params={"filter": "country = 'DE'", "filter-lang": "cql2-text"},
)
check(r, "CQL2-text filter (country=DE)")
fc = r.json()
names = [f["properties"]["name"] for f in fc.get("features", [])]
print(f"Matching items: {names}")
assert all(f["properties"]["country"] == "DE" for f in fc.get("features", [])), \
    "Filter did not restrict to DE"

In [ ]:
# Filter: population > 3000000 using CQL2-text
r = client.get(
    f"/features/catalogs/{CATALOG_ID}/collections/{COLL_ID}/items",
    params={"filter": "population > 3000000", "filter-lang": "cql2-text"},
)
check(r, "CQL2-text filter (population > 3M)")
fc = r.json()
names = [f["properties"]["name"] for f in fc.get("features", [])]
print(f"Cities with population > 3M: {names}")

## 5. CQL2-JSON filter

In [ ]:
# CQL2-JSON compound filter: population > 2000000 AND country IN ('FR','ES')
cql_filter = {
    "op": "and",
    "args": [
        {"op": ">", "args": [{"property": "population"}, 2000000]},
        {"op": "in", "args": [{"property": "country"}, ["FR", "ES"]]},
    ]
}
r = client.get(
    f"/features/catalogs/{CATALOG_ID}/collections/{COLL_ID}/items",
    params={"filter": json.dumps(cql_filter), "filter-lang": "cql2-json"},
)
check(r, "CQL2-JSON compound filter")
fc = r.json()
names = [f["properties"]["name"] for f in fc.get("features", [])]
print(f"Matching: {names}")

## 6. Queryables

In [ ]:
# GET /features/catalogs/{cat}/collections/{col}/queryables
r = client.get(f"/features/catalogs/{CATALOG_ID}/collections/{COLL_ID}/queryables")
check(r, "Get queryables")
q = r.json()
print(f"Schema type: {q.get('type')}  properties: {list(q.get('properties', {}).keys())[:8]}")

## Teardown

In [ ]:
# Delete single item
r = client.delete(f"/features/catalogs/{CATALOG_ID}/collections/{COLL_ID}/items/{ITEM_ID}")
check(r, "Delete item", expected=(204,))

# Delete collection
r = client.delete(f"/features/catalogs/{CATALOG_ID}/collections/{COLL_ID}")
check(r, "Delete collection", expected=(204,))

# Delete catalog
r = client.delete(f"/stac/catalogs/{CATALOG_ID}")
check(r, "Delete catalog", expected=(204, 200))

client.close()
print("Teardown complete.")